# 04_lakebase
Captured 2026-11-21 23:49 UTC

**Operational write path with Part 11-style controls.** Planners propose, entitled approvers decide with a rationale, no one approves their own proposal, every change lands in an append-only audit log, decisions cannot be deleted, and Postgres RLS mirrors the UC region rules (an EU planner cannot write or see US actions). Decisions land back in Delta (gold.mitigation_decision) under the same region filter.


In [1]:
-- code: approval rules and region RLS (src/serve/lakebase_setup.sql)
# src/serve/lakebase_setup.sql
CREATE OR REPLACE FUNCTION decisions.fn_check_decision() RETURNS trigger LANGUAGE plpgsql AS $$
DECLARE who TEXT := current_setting('app.user', true);
BEGIN
  IF TG_OP = 'UPDATE' AND NEW.status IS DISTINCT FROM OLD.status THEN
    IF OLD.status <> 'PROPOSED' THEN RAISE EXCEPTION 'decision already recorded for action %', OLD.action_id; END IF;
    IF NOT EXISTS (SELECT 1 FROM ops.user_entitlement e WHERE e.principal = who AND e.can_approve) THEN
      RAISE EXCEPTION '% is not entitled to approve mitigations', who; END IF;
    IF who = NEW.proposed_by OR who = NEW.on_behalf_of THEN
      RAISE EXCEPTION 'segregation of duties: % cannot decide an action they proposed', who; END IF;
    IF coalesce(trim(NEW.decision_rationale), '') = '' THEN RAISE EXCEPTION 'decision rationale is required'; END IF;
    NEW.decided_by := who;
    NEW.decided_at := now();
  END IF;
  RETURN NEW;
END $$;

# src/serve/lakebase_setup.sql
CREATE POLICY region_access ON decisions.mitigation_action
  USING (EXISTS (SELECT 1 FROM ops.user_entitlement e WHERE e.principal = current_setting('app.user', true)
                 AND (e.region = '*' OR e.region = mitigation_action.region)))
  WITH CHECK (EXISTS (SELECT 1 FROM ops.user_entitlement e WHERE e.principal = current_setting('app.user', true)
                 AND (e.region = '*' OR e.region = mitigation_action.region)));

tests below run against this

In [2]:
-- connected as a service principal role without BYPASSRLS
SELECT current_user, rolbypassrls

[["7739855e-67af-424e-a487-b03d30433d3a", false]]

In [3]:
-- synced tables from UC [app.user=maggie.barrett@databricks.com]
SELECT (SELECT count(*) FROM ops.disruption_impact) AS impact_rows, (SELECT count(*) FROM ops.user_entitlement) AS entitlements

[[38, 2]]

In [4]:
-- planner proposes for EU [app.user=7739855e-67af-424e-a487-b03d30433d3a]
INSERT INTO decisions.mitigation_action (disruption_id, sku_id, region, action_type, description, proposed_by, proposed_source) SELECT disruption_id, sku_id, region, 'REALLOCATE_INVENTORY', 'Evidence test: move 4 weeks of EU stock', '7739855e-67af-424e-a487-b03d30433d3a', 'PLANNER' FROM ops.disruption_impact WHERE region = 'EU' ORDER BY supply_gap_days DESC LIMIT 1 RETURNING action_id, region, status

[[7, "EU", "PROPOSED"]]

In [5]:
-- planner tries to propose for US (RLS WITH CHECK) [app.user=7739855e-67af-424e-a487-b03d30433d3a]
INSERT INTO decisions.mitigation_action (disruption_id, region, action_type, description, proposed_by, proposed_source) VALUES ('test', 'US', 'ALLOCATE_DEMAND', 'should be blocked', '7739855e-67af-424e-a487-b03d30433d3a', 'PLANNER')

ERROR (expected): new row violates row-level security policy for table "mitigation_action"

In [6]:
-- planner sees only EU actions [app.user=7739855e-67af-424e-a487-b03d30433d3a]
SELECT region, count(*) FROM decisions.mitigation_action GROUP BY region

[["EU", 4]]

In [7]:
-- planner tries to approve (not entitled) [app.user=7739855e-67af-424e-a487-b03d30433d3a]
UPDATE decisions.mitigation_action SET status='APPROVED', decision_rationale='x' WHERE status='PROPOSED' AND region='EU'

ERROR (expected): 7739855e-67af-424e-a487-b03d30433d3a is not entitled to approve mitigations

In [8]:
-- approval without rationale [app.user=maggie.barrett@databricks.com]
UPDATE decisions.mitigation_action SET status='APPROVED' WHERE status='PROPOSED' AND region='EU'

ERROR (expected): decision rationale is required

In [9]:
-- exec approves with rationale [app.user=maggie.barrett@databricks.com]
UPDATE decisions.mitigation_action SET status='APPROVED', decision_rationale='Stock exists in EU-DE1; approved for 4 weeks' WHERE action_id = (SELECT max(action_id) FROM decisions.mitigation_action WHERE status='PROPOSED' AND region='EU') RETURNING action_id, status, decided_by

[[7, "APPROVED", "maggie.barrett@databricks.com"]]

In [10]:
-- audit trail is append-only [app.user=maggie.barrett@databricks.com]
DELETE FROM decisions.audit_log

ERROR (expected): permission denied for table audit_log

In [11]:
-- decisions cannot be deleted [app.user=maggie.barrett@databricks.com]
DELETE FROM decisions.mitigation_action

ERROR (expected): permission denied for table mitigation_action

In [12]:
-- audit trail [app.user=maggie.barrett@databricks.com]
SELECT action_id, event, actor, at FROM decisions.audit_log ORDER BY audit_id DESC LIMIT 6

[[7, "UPDATE:APPROVED", "maggie.barrett@databricks.com", "2026-11-21 23:49:39.171132+00:00"], [7, "INSERT:PROPOSED", "7739855e-67af-424e-a487-b03d30433d3a", "2026-11-21 23:49:38.697299+00:00"], [5, "UPDATE:APPROVED", "maggie.barrett@databricks.com", "2026-11-21 23:40:29.372243+00:00"], [5, "INSERT:PROPOSED", "7739855e-67af-424e-a487-b03d30433d3a", "2026-11-21 23:40:28.831037+00:00"], [3, "UPDATE:APPROVED", "maggie.barrett@databricks.com", "2026-11-21 22:52:28.975991+00:00"], [3, "INSERT:PROPOSED

In [13]:
-- decisions land back in governed Delta (as of the last land_decisions job run)
SELECT ... FROM supply_chain.gold.mitigation_decision

action_id | region | action_type          | status   | decided_by                  
---------------------------------------------------------------------------------------
1         | EU     | REALLOCATE_INVENTORY | APPROVED | maggie.barrett@databricks.co
(1 rows)
{
  "task": "land_decisions",
  "actions_merged": 1,
  "audit_rows_appended": 2,
  "status_counts": {
    "APPROVED": 1
  }
}